Install `Mastodon.py`, which allows Python to interact with the Mastodon API.

In [ ]:
!pip install Mastodon.py


Import `Mastodon` to interact with the Mastodon API, along with `csv` to write our results to a CSV file and `datetime` to work with dates.

I also import `getpass` to prompt for my Mastodon API access token. Authentication is needed to search the contents of posts rather than just hashtags. Using `getpass()` allows me to enter my token without displaying it in the notebook.

#### Getting a Mastodon API access token

To use the Mastodon search API, you'll first need to create an account on [mastodon.social](https://mastodon.social).

Once logged in, go to Preferences → Development and create a new application. You can give it a name. I used the class name, SDS 510.

Under Scopes, select these permissions:

- `read:accounts`
- `read:search`
- `read:statuses`

Once the application is created, copy its access token. You don't need the Client Key or Client Secret.

When the notebook runs, `getpass()` prompts you to enter your access token. I chose this approach rather than storing the token in the notebook so it wouldn't be published when I upload the notebook to GitHub.

Another option would be to store the token in a separate `.env` file, but that would require additional setup. Using `getpass()` is simpler for this assignment and keeps the token out of the notebook's saved code and displayed output.

In [ ]:
from mastodon import Mastodon
import csv
import datetime

In [ ]:
from getpass import getpass

mastodon = Mastodon(
    access_token=getpass('Mastodon access token: '),
    api_base_url='https://mastodon.social'
)

The keywords are the five terms provided in the assignment: cheese, pickles, buns, meat, and sesame.

I use `datetime` to calculate the cutoff date, which is six weeks before the current date. This establishes how far back we want to collect posts.

In [ ]:
keywords = ['cheese', 'pickles', 'buns', 'meat', 'sesame']  
cutoff = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(weeks=6)  

First, I create an empty list called `records` to store the posts collected from Mastodon.

The `for` loop goes through each of the five keywords. For each keyword, the query includes the keyword and the `after:` date restriction so Mastodon searches for posts after the cutoff date.

`mastodon.search()` searches the contents of posts and requests up to 40 results at a time. The `page_number` variable starts at zero so I can track how many pages have been retrieved.

The `while` loop continues as long as there are results and the results contain statuses. `results` contains the response from Mastodon, while `results['statuses']` contains the actual posts returned by the search.

For each post, I check its `created_at` date to make sure it falls within the six-week period. I also check whether the keyword actually appears in `eachpost['content']`. I added this check because I found that Mastodon was sometimes returning posts that didn't contain the keyword in either the post text or hashtags. Using `.lower()` makes the comparison case-insensitive, so capitalization doesn't affect whether a post matches.

For posts that meet both conditions, I create a record containing the keyword, date, author, text, URL, and counts for replies, reblogs, and favorites. Each record is added to the `records` list. I used `strftime` on the date to ensure it's human-readable.

`mastodon.fetch_next()` retrieves the next page of results, and the loop continues until there are no more pages.

I also added a `print()` statement to display the keyword and page number while the search is running. I found this useful for longer searches so I could see that the code was still working and how far it had progressed. You can comment that line out if you do not wish to see it.

The module's example uses a 10-page limit with `page_number < 10`. I left that limit out because I wanted to collect the available results for the full six-week period.

To limit the text search to 10 pages for testing, change the `while` loop to:

```python
while results and results['statuses'] and page_number < 10:
```

Removing `and page_number < 10` allows the search to continue until no more results are returned.

In [ ]:
records = []

for keyword in keywords:
    query = f"{keyword} after:{cutoff.date()}"
    results = mastodon.search(query, result_type='statuses', limit = 40)
    page_number = 0
    
    while results and results['statuses']:
        page_number += 1
        for eachpost in results['statuses']:
            if eachpost['created_at'] < cutoff:
                continue
            if keyword.lower() not in eachpost['content'].lower():
                continue
            record = {
                'keyword':keyword, 
                'date': eachpost['created_at'].strftime('%Y-%m-%d %H:%M:%S'),
                'author': eachpost['account']['acct'],
                'text': eachpost['content'],
                'url': eachpost['url'],
                'replies': eachpost['replies_count'],
                'reblogs': eachpost['reblogs_count'],
                'favorites': eachpost['favourites_count'],
            }
            records.append(record)

        results = mastodon.fetch_next(results)
        print(f"Text search: {keyword}, page {page_number}")



The next section searches hashtags using `mastodon.timeline_hashtag()`, which is the approach demonstrated in the module's lecture.

I loop through the same five keywords, requesting up to 40 posts at a time and resetting `page_number` to zero for each keyword.

The `while` loop continues through the available pages. As with the text search, I check that each post is newer than the six-week cutoff and that its content contains the keyword.

Posts that meet those conditions are added to the same `records` list, using the same fields as the text search.

I included another `print()` statement to show which hashtag is being searched and the current page number.  You can comment that line out if you do not wish to see it.

To limit the hashtag search to 10 pages for testing, change the `while` loop to:

```python
while posts and page_number < 10:
```

Removing `and page_number < 10` allows the hashtag search to continue until no more pages are returned.

In [ ]:
for keyword in keywords:
    posts = mastodon.timeline_hashtag(keyword, limit=40)
    page_number = 0

    while posts:
        page_number += 1

        for eachpost in posts:
            if eachpost['created_at'] < cutoff:
                continue
            if keyword.lower() not in eachpost['content'].lower():
                continue

            record = {
                'keyword': keyword,
                'date': eachpost['created_at'].strftime('%Y-%m-%d %H:%M:%S'),
                'author': eachpost['account']['acct'],
                'text': eachpost['content'],
                'url': eachpost['url'],
                'replies': eachpost['replies_count'],
                'reblogs': eachpost['reblogs_count'],
                'favorites': eachpost['favourites_count'],
            }
            records.append(record)

        posts = mastodon.fetch_next(posts)
        print(f"Hashtag: {keyword}, page {page_number}")

Finally, I use `csv.DictWriter()` to save the collected records to `mastodon-hits.csv`.

The columns are keyword, date, author, text, URL, replies, reblogs, and favorites. `writeheader()` writes the column names, and `writerows(records)` writes the collected posts to the CSV file.

In [ ]:
with open('mastodon-hits.csv', 'w') as f:
    columns = ['keyword', 'date', 'author', 'text', 'url', 'replies', 'reblogs', 'favorites']
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(records)